In [ ]:

import pandas as pd
import duckdb

# 投料数据：Sakila 门店真实行为流
dvd_rentals = pd.DataFrame({
    'rental_id': [1001, 1002, 1003, 1004],
    'user_id': [501, 502, 503, 504],
    'rental_date': ['2026-06-15 09:00:00', '2026-06-16 14:20:00', '2026-06-17 11:00:00', '2026-06-18 10:00:00'],
    'return_date': ['2026-06-19 11:30:00', '2026-06-16 16:20:00', '2026-06-22 14:00:00', None] # 🛡️ 注意：包含未归还的坑（None）
})

### 🎯 2. 核心刚性需求

1. **统一时间轴**：将 `rental_date` 和 `return_date` 刚性转化为时间类型。
    
2. **计算预期归还时间**：门店规定，所有碟片的**刚性租借期限一律为 3 天**（即 `rental_date` 往后平移 3 天）。
    
3. **计算逾期秒数**：用实际归还时间（`return_date`）减去预期归还时间。如果没还（`None`），或者没有逾期（计算结果 $\le 0$ 秒），**则这门课的高级函数不需要处理它（在结果中过滤掉）**。
    
4. **最终输出**：只需输出已经发生逾期的 `user_id`、`rental_date`、`return_date` 以及具体的逾期秒数 `overdue_seconds`。

In [6]:
# =====================================================================
# ⚔️  轨道一：PostgreSQL 
# =====================================================================
sql_query = """
WITH format_conversion_stage AS (
    SELECT  user_id,
            rental_date :: TIMESTAMP AS rental_date,
            return_date :: TIMESTAMP AS return_date,
            rental_date :: TIMESTAMP + INTERVAL '3 days' AS expected_return_date
    FROM dvd_rentals
),
filter_stage AS (
    SELECT  user_id,
            rental_date,
            return_date,
            (return_date - expected_return_date) AS overdue_interval
    FROM format_conversion_stage
    WHERE   return_date IS NOT NULL
      AND   return_date > expected_return_date
)
SELECT  user_id,
        rental_date,
        return_date,
        EXTRACT(EPOCH FROM overdue_interval) :: INTEGER AS overdue_seconds
FROM filter_stage
ORDER BY user_id;
"""
df_sql = duckdb.query(sql_query).df()
print(df_sql)

   user_id         rental_date         return_date  overdue_seconds
0      501 2026-06-15 09:00:00 2026-06-19 11:30:00            95400
1      503 2026-06-17 11:00:00 2026-06-22 14:00:00           183600


In [11]:
# =====================================================================
# ⚔️  轨道二：PANDAS 
# =====================================================================
df_pandas_pro = (
    dvd_rentals.assign(
        # 1. 刚性强转时间轴坐标（assign 算子可以在不改变原 df 的情况下注入/修改列）
        rental_date=pd.to_datetime(dvd_rentals['rental_date']),
        return_date=pd.to_datetime(dvd_rentals['return_date']),
        # 2. 物理平移，精准锁定预期归还时间轴
        expected_return_date=lambda x: pd.to_datetime(x['rental_date']) + pd.Timedelta(days=3)
    )
    # 3. 刚性拦截闸：用 query 配合 Python 原生引擎，零外部变量，一条龙过滤未归还与未逾期
    .query(
        "return_date.notna() and return_date > expected_return_date", 
        engine='python'
    )
    # 4. 降维压榨秒数：用 assign 计算最终特征，并刚性强转 ::INTEGER
    .assign(
        overdue_seconds=lambda x: (x['return_date'] - x['expected_return_date'])
                                  .dt.total_seconds()
                                  .astype(int)
    )
    # 5. 矩阵裁剪与对齐：只保留下游模型需要的特征字段，并按 user_id 刚性排序
    [['user_id', 'rental_date', 'return_date', 'overdue_seconds']]
    .sort_values(by='user_id')
    .reset_index(drop=True)
)

print("🐼 顶级专业 Pandas 轨道特征输出：")
print(df_pandas_pro)

🐼 顶级专业 Pandas 轨道特征输出：
   user_id         rental_date         return_date  overdue_seconds
0      501 2026-06-15 09:00:00 2026-06-19 11:30:00            95400
1      503 2026-06-17 11:00:00 2026-06-22 14:00:00           183600


In [12]:

# =====================================================================
# 🚨 终极对账大闸
# =====================================================================
pd.testing.assert_frame_equal(
    df_sql.reset_index(drop=True), 
    df_pandas_pro.reset_index(drop=True),
    check_dtype=False
)
print("🏆【完美会师！】 SQL 轨道与工业级 Pandas 链式轨道彻底像素级对齐！无懈可击！")

🏆【完美会师！】 SQL 轨道与工业级 Pandas 链式轨道彻底像素级对齐！无懈可击！
